# 03 - ML Models
Logistic Regression e XGBoost com MLflow tracking.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!pip install pyspark==3.5.0 delta-spark==3.2.0 mlflow==2.10.0 xgboost -q

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FraudDetection-ML")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE_PATH     = "/content/lakehouse"
GOLD_PATH     = f"{BASE_PATH}/gold/ml_dataset"
PIPELINE_PATH = f"{BASE_PATH}/pipeline"
MODELS_PATH   = f"{BASE_PATH}/models"
os.makedirs(MODELS_PATH, exist_ok=True)

print(f"Spark {spark.version} pronto!")

In [ ]:
from pyspark.sql.functions import col, when

df = spark.read.format("delta").load(GOLD_PATH)

# Peso para tratar desbalanceamento
# Fraudes recebem peso proporcional à sua raridade
fraud_count   = df.filter(col("label") == 1).count()
legit_count   = df.filter(col("label") == 0).count()
total         = fraud_count + legit_count
fraud_weight  = total / (2 * fraud_count)
legit_weight  = total / (2 * legit_count)

print(f"Fraudes : {fraud_count:,} | peso = {fraud_weight:.2f}")
print(f"Legitimas: {legit_count:,} | peso = {legit_weight:.2f}")

df = df.withColumn(
    "weight",
    when(col("label") == 1, fraud_weight).otherwise(legit_weight)
)

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Treino: {train_df.count():,} | Teste: {test_df.count():,}")

## Logistic Regression

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import mlflow
import mlflow.spark

mlflow.set_experiment("fraud-detection")

auc_evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

def recall_score(predictions):
    tp = predictions.filter((col("label") == 1) & (col("prediction") == 1)).count()
    fn = predictions.filter((col("label") == 1) & (col("prediction") == 0)).count()
    return tp / (tp + fn) if (tp + fn) > 0 else 0

with mlflow.start_run(run_name="logistic_regression"):
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        weightCol="weight",
        maxIter=20
    )
    model_lr = lr.fit(train_df)
    predictions_lr = model_lr.transform(test_df)

    auc    = auc_evaluator.evaluate(predictions_lr)
    recall = recall_score(predictions_lr)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("maxIter", 20)
    mlflow.log_param("weightCol", True)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("recall", recall)
    mlflow.spark.log_model(model_lr, "model")

    model_lr.save(f"{MODELS_PATH}/logistic_regression")

    print(f"AUC   : {auc:.4f}")
    print(f"Recall: {recall:.4f}")

predictions_lr.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

## XGBoost

In [ ]:
from xgboost.spark import SparkXGBClassifier

with mlflow.start_run(run_name="xgboost"):
    xgb = SparkXGBClassifier(
        features_col="features",
        label_col="label",
        weight_col="weight",
        prediction_col="prediction",
        probability_col="probability",
        max_depth=6,
        eta=0.1,
        num_round=100
    )
    model_xgb = xgb.fit(train_df)
    predictions_xgb = model_xgb.transform(test_df)

    auc    = auc_evaluator.evaluate(predictions_xgb)
    recall = recall_score(predictions_xgb)

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("eta", 0.1)
    mlflow.log_param("num_round", 100)
    mlflow.log_param("weightCol", True)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("recall", recall)
    mlflow.spark.log_model(model_xgb, "model")

    model_xgb.save(f"{MODELS_PATH}/xgboost")

    print(f"AUC   : {auc:.4f}")
    print(f"Recall: {recall:.4f}")

predictions_xgb.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

## Comparacao de modelos via MLflow

In [ ]:
import pandas as pd

runs = mlflow.search_runs(experiment_names=["fraud-detection"])

print("Resultados registrados no MLflow:")
print(
    runs[[
        "tags.mlflow.runName",
        "metrics.auc",
        "metrics.recall",
        "params.model"
    ]].to_string(index=False)
)